<a href="https://colab.research.google.com/github/bharathvaddineniK/agenticAi/blob/main/monitoring_tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import os
from openai import OpenAI
from google.colab import userdata
import json
import requests
import random
import time

In [15]:
client = OpenAI(api_key = userdata.get('OPENAI_APIKEY'))

In [70]:
def mock_health_check():
  outcome = random.choice(['healthy', 'timeout', 'unhealthy', 'error'])
  if outcome == 'healthy':
    print(f"\n[INFO] - service is healthy")
    return {'status': 'healthy'}, 200
  elif outcome == 'timeout':
    print(f"\n[WARNING] - service is timedout")
    time.sleep(10)
    return {'status': 'timeout'}, 408
  elif outcome == 'unhealthy':
    print(f"\n[ERROR] - service is unhealthy")
    return {'status': 'unhealthy'}, 500
  else:
    print(f"\n[ERROR] - service is in error state")
    return {'status': 'error'}, 500


In [47]:
mock_health_check()
#


[ERROR] - service is in error state


({'status': 'error'}, 500)

In [71]:
def mock_restart_serivce():
  print(f"service restarting.....")
  time.sleep(5)
  print(f"service restarted successfully")

In [72]:
mock_restart_serivce()

service restarting.....
service restarted successfully


In [73]:
AVAILABLE_TOOL_NAMES = {
    "health_check": mock_health_check,
    "restart_service": mock_restart_serivce
}

In [74]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "health_check",
            "description": "check the health of the service",
            "parameters": {
                "type": "object",
                "properties":{

                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "restart the service",
            "parameters": {
                "type": "object",
                "properties":{

                },
                "required": []
                }
        }
    }
]

In [77]:
import json
import time

def monitor_agent(userprompt):

    messages = [
        {
            "role": "system",
            "content": """
You are a service monitoring agent. Your job is to continuously monitor the health of a service and automatically recover it if necessary.

Your goal:
Ensure the service stays healthy. If the service fails three consecutive health checks, you must restart it.

Available tools:
- health_check: checks the health of the service and returns a status_code
- restart_service: restarts the service

Rules:
- A healthy service returns status code 200.
- Any status code other than 200 counts as a failure.
- You must track the number of consecutive failures.
- If a health check returns 200, reset the failure counter to 0.
- If the health check returns a non-200 status code, increase the failure counter.
- If the failure counter reaches 3, call the restart_service tool and reset the counter.

Workflow:
1. Call the health_check tool.
2. Inspect the returned status_code.
3. Print the health check result. In case of healthy print [INFO] - System healthy. In case of failure print [ERROR] - System unhealthy.
4. Update the failure counter accordingly.
5. If the failure counter reaches 3, call restart_service.
6. Continue monitoring the service.

Important:
- Do not restart the service unless there are three consecutive failures.
- Always check the health of the service before deciding on an action.
- Do not stop health_check even after multiple continuous healthy responses.
"""
        },
        {"role": "user", "content": userprompt}
    ]

    while True:

        print("\nAI monitoring...")

        response = client.chat.completions.create(
            model="gpt-4o",
            tools=tools_schema,
            messages=messages,
            tool_choice="auto"
        )

        response_msg = response.choices[0].message
        messages.append(response_msg)

        # If the model wants to call a tool
        if response_msg.tool_calls:

            for tool_call in response_msg.tool_calls:

                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments or "{}")

                function_to_call = AVAILABLE_TOOL_NAMES.get(func_name)

                if function_to_call:

                    func_response = function_to_call(**func_args)

                    messages.append(
                        {
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": func_name,
                            "content": json.dumps(func_response)
                        }
                    )

        else:
            print("\nAI: Monitoring tool not called, continuing monitoring")

        # Wait before next health check
        time.sleep(1)

In [78]:
monitor_agent("Monitor payment service")


AI monitoring...

[WARNING] - service is timedout

AI monitoring...

[ERROR] - service is in error state

AI monitoring...

[ERROR] - service is in error state

AI monitoring...
service restarting.....
service restarted successfully

AI monitoring...

[ERROR] - service is unhealthy

AI monitoring...

[WARNING] - service is timedout

AI monitoring...

[INFO] - service is healthy

AI monitoring...

AI: Monitoring tool not called, continuing monitoring

AI monitoring...

[ERROR] - service is in error state

AI monitoring...

AI: Monitoring tool not called, continuing monitoring

AI monitoring...

[ERROR] - service is unhealthy

AI monitoring...

[ERROR] - service is in error state

AI monitoring...
service restarting.....
service restarted successfully

AI monitoring...

AI: Monitoring tool not called, continuing monitoring

AI monitoring...

[ERROR] - service is unhealthy

AI monitoring...

AI: Monitoring tool not called, continuing monitoring

AI monitoring...

[INFO] - service is heal

KeyboardInterrupt: 